## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> 1. 도입 </span>

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 1.1 RNN의 한계 & 어텐션의 등장 배경 </span>

1. **RNN(Recurrental Neural Network) 한계**
   - **개념** : <span style="background-color:pink">앞-뒤 내용이 서로 연관된 데이터(ex.순차 데이터)를 처리하기 위해 기억을 통해 이전 단계에서 처리한 정보를 현재 단계의 입력에 함께 활용하는 신경망</span>
     - **구조** : $X$(출력레이어 - 은닉레이어 - 출력레이어), $W$, $b$
     - 기억의 구조 -> $X_{t-1}$의 입력에 대해 $O_{t-1}$의 출력이 나오고, $X_{t-1}$시점의 정보 $h_{t-1}$은 다음 시점의 $h_t$로 전달되어 과거 정보가 계속 전달된다.
   - **핵심 원리 : <span style="background-color:yellow">가중치 공유!</span>** (모든 타임 스텝에서 동일한 가중치를 반복해 사용)
     - 가중치 공유의 이점 : 파라미터의 효율성, 학습의 일반화
   - **한계** : **장기 의존성 문제 : 기울기 폭주&소실 문제**
     - 즉, 초반부 데이터들이 후반부 데이터들에게 과도하게 반영되거나, 거의 반영되지 않는 문제!
   - **극복을 위한 노력** : <span style="background-color:yellow">**LSTM 모델**</span> (중요한 것은 오래 기억 + 필요없는 건 잊기!)<br>
     - **중요한 정보** : `Input gate` -> `Cell State` : 가중치 곱 거치지 X 직선적으로 전달됨
       - 이때 Output에 사용할 정보는 `Output gate`로 가져와 출력!
     - **잊을 정보** : **3가지 GATE** : `Input gate` -> `Forget gate`
     - **장단점**
       - **장점** : 오래전에 입력된 정보도 Cell State로 오래 기억 가능 / gradient 유지 가능 / 정보 선택 가능
       - **단점** : 게이트가 3개라 계산이 복잡 + 시간이 오래 걸림 / 긴 문장에서는 기울기 소실 문제 여전함

2. **<span style="background-color:yellow">Seq2Seq</span>**
   - **개념** : <span style="background-color:pink">input 내용을 다른 시퀀스로 변환하는 모델 구조</span>
   - **장점** : 입출력 단어 개수가 달라도 사용 가능, 다양한 변환 작업에 적용 가능
   - **구조** : `Encoder`에서 입력데이터를 순서대로 읽고 중요한 정보를 `Context Vector`로 표현 <br> -> `Decoder`는 `Context Vector`를 바탕으로 출력 시퀀스를 생성
   - **병목현상** 
     - 입력데이터가 길어질 수록 `Context Vector`에 모든 정보를 담기 어려움 -> **중요한 정보가 손실**
     - Decoder는 **매번 같은 `Context Vector`만 참고** -> 출력 단어마다 필요한 정보를 다르게 반영하지 X
     - <span style="background-color:yellow">**해결사 : Attention!**</span>
       - `Decoder`가 단어 생성할 때마다 입력 문장 전체를 다시 봄 <br> -> 현재 단어와 관련 있는 부분에 더 집중해 새로운 `Context Vector` 생성

3. **Attention**
   - **개념** : <span style="background-color:pink">중요한 정보에 더 집중하는 매커니즘</span>
     - **현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여해 활용**함
     - 출력 단어 생성 시점 마다 새로운 `Context Vector`를 동적으로 계산(입력 문장 속 단어들 중 어떤 단어에 얼마나 집중할지 결정)하는 기술! -> **병렬화**
   - **과정**
      1. 입력을 숫자 벡터 형태로 변환
      2. 단어별 중요도 계산
      3. 중요도에 따른 입력 정보 조합
   - **장점**
     - 시간 경과에 대한 유연성
       - 시간적(순서적) 거리에 제약을 받지 않음 -> RNN의 장기 의존성 문제 타파!!
     - 공간에 대한 유연성
       - 고정된 필터를 사용하는 CNN과 달리, 픽셀 간 거리에 상관없이 한 번에 전체 이미지의 전역적 관계 학습 가능
     - 병렬화
       - self-attention에서 모든 단어의 관계를 한 번에, 독립적으로 계산 가능!
     - LLM뿐만 아니라 이미지 생성의 확산 모델이나 비전 분야의 물체 감지, 이미지 분할 등 다양한 작업에서 좋은 성능을 발휘!

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2. 어텐션의 개념 </span>
#### 1. **Attention is All You Need (2017)**
> Transformer 구조를 처음 발표한 구글 브레인 논문
> 이 논문 발표 이후, Attention을 쓰던 딥러닝 모델들이 **Self-Attention** 방식을 채택하게 됨

1. **논문의 개요** 
- **연구 배경** : RNN, RNN기반 모델들의 한계점인 느린 학습 속도, 장기 의존성 문제
  - *"순차적 계산 구조 제거 + 인코더부터 디코더까지 Attention만을 사용하는 모델을 만든다면..?"*
- **Transformer의 등장**
  - RNN의 순환 구조 제거 + Self-attention에 기반한 Transformer 모델 제안
- **발견 내용**
  - 성공적 모델링에 Recurrence & Convolution이 필수가 아니다 <br> -> **순환 구조 없이 Attention만으로 좋은 성능** 낼 수 있음을 증명
  - **병렬화**가 가능해짐
  - 이를 통해 *학습 속도의 획기적인 단축* + *거대 데이터를 다루는 모델 제작*이 가능해짐

2. **논문의 파급 효과**
   - LLM의 탄생 (BERT, GPT)
   - NLP 평정 (거의 모든 NLP 분야에서 트랜스포머 기반 모델들이 표준이 됨)
   - NLP 외에 컴퓨터 비전, 음성 처리, 신약 개발 등으로 확장됨

#### 2. <span style="background-color:yellow">**Transformer**</span>
1. **개념**
   - <span style="background-color:pink">RNN의 순차적인 계산 방식을 제거하고, **Attention만으로 문장의 의미+구조를 파악**하는 모델</span>
   - **이전 모델들과의 비교**
     - 기존의 Attention기반 Seq2Seq : RNN으로 인코더+디코더 구현 + attention 추가한 구조
     - **Transformer : Attention만으로 인코더+디코더 구현!!**

2. **핵심 기술**
   - **<span style="background-color:yellow">Self-Attention</span>**
     - 문장 안에서 **어떤 단어가 다른 단어들과 얼마나 중요한 관계**를 맺고 있는지 한 번에 파악
   - **<span style="background-color:yellow">Multi-Head Attention</span>**
     - Self-Attention을 **여러 개의 머리로 동시에 서로 다른 관점**에서 실행
   - **<span style="background-color:yellow">Positional Encoding</span>**
     - 단어의 **위치 정보를 벡터에 추가**해 각 단어가 **문장의 몇번째 위치에 존재**하는지 알 수 있음

## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> 2. Self-attention </span>

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2.1 Query, Key, Value </span>
> Attention의 기본 구조

1. **Q (Query)** : 질문, 요청
   1. 지금 당장 알고 싶거나, 초점을 맞추고 있는 대상
   2. 어텐션에서는, **나 자신이 쿼리**가 됨

2. **K (Key)** : 검색 대상이 되는 정보들이 달고 있는 이름표, 색인
   1. 쿼리가 자신이 다른 것들과 얼마나 관련이 있는지 측정할 때 비교하는 대상
   2. 어텐션에서는, **문장 내 모든 단어들이 각각 키의 역할**을 수행, 현재 쿼리 단어와 얼마나 관련이 있는지 평가를 진행

3. **V (Value)** : 실제 내용물
   1. 키와 한 쌍으로 묶여 있음
   2. 어텐션에서는, 특정 단어(Q)가 다른 단어(K)와 관련이 깊다고 판단되면, **그 단어(K)가 가진 실제 의미 정보(V)를 가져와 자신의 의미를 보강**하는 데 사용

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2.2 Cross-Attention vs Self-Attention </span>
#### **1. 기존 Attention** (in Seq2Seq, Cross-Attention)
    > 두개의 서로 다른 정보 소스를 연결하는 다리 역할! <br>

   - <span style="background-color:yellow">**참조 대상 : 다른 시퀀스** (주로 디코더가 인코더의 정보를 참고할 때 사용)</span>
   - **핵심 과정**
     1. **쿼리 만들기** : 지금까지 생성된 단어들을 보고 현재 상태를 하나의 벡터로 요약
     2. **키, 벨류 불러오기** : 인코더는 입력 문장을 처리해 각 단어에 대한 벡터를 가지고 있음
     3. **유사도** 계산 : 디코더의 쿼리 - 인코더의 각 키 유사도(관련성) 계산<br>    -> 유사도 점수를 **softmax**로 정규화 -> 전체가 1이 되도록 **가중치로 변환**
     4. 가중치 합계 (**Weighted Sum**) : 인코더의 벨류들을 가중치로 섞어 하나의 벡터(context vector)로 만듦
     5. 다음 단어를 **예측** by Decoder : 자기 자신의 정보 + 방금 만든 context vector를 활용해 예측

#### **2. Self-Attention**
   - <span style="background-color:yellow">**참조 대상** : 문장 자체, 즉 나 자신 -> 문장 스스로가 문맥을 이해하는 과정임</span>
   - **핵심 과정**
     1. **단어의 프로필 만들기 (벡터 임베딩)** : 각 단어의 고유한 특징을 담은 벡터로 변환
     2. **단어 간 관계 점수 계산** (내정 및 정렬 점수) : 모든 단어 벡터끼리 서로 얼마나 유사한지 관계점수 계산
     3. **중요도 배분** (소프트맥스 & 어텐션 가중치) : 계산된 관계 점수를 총합이 1이 되는 Attention Weight로 변환<br> -> 특정 단어를 이해하는 데 다른 단어들이 각각 몇%씩 중요한가?를 보여줌
     4. **새로운 벡터 생성** : 문장 내 다른 단어들의 정보를 자신에데 맞게 조합 -> 문맥이 반영된 새로운 프로필(벡터)로 생성됨
   - **상세 설명**
     - 학습 이전에는 트랜스포머 모델이 최적의 벡터 임베딩, 정렬점수를 어떻게 생성할지 모름
     - 학습을 통해 학습데이터에서 추출한 예시들을 기반으로 예측을 수행, 손실함수는 각 예측의 오차를 정량화
     - 예측, 역전파, 경사하강법을 통해 모델 가중치 업데이트 반복
     - 이 과정을 거쳐 정확한 출력을 생성하는 **벡터 임베딩, 정렬 점수, 어텐션 가중치를 학습**하게 됨!

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 2.3 Scaled Dot-Product Attention </span>
#### **1. Dot-Product Attention**
- **개념** : **쿼리와 키 벡터를 내적해 유사도**를 구하는 방법 (Attention Score를 계산하는 가장 기본적인 방법)
- **과정**
    1. **쿼리(Q) 벡터 - 모든 키(K) 벡터를 각각 내적**해 **유사도** 계산 진행
        → Attention Score = Q · K    
    2. 계산된 값에 **Softmax** 함수 적용 -> 총합이 1인 **Attention Weight** 산출
    3. 이 **가중치를 벨류 벡터에 곱해** 최종 결과값 계산
- **문제 : Attention Score의 극단화**
  - 벡터차원이 커질 수록 내적값이 극단적으로 커지거나 작아지는데, 이런 값이 Softmax에 입력되며 특정 단어에 확률이 과다 집중<br> -> gradient가 0에 수렴하게 되는 Vanishing Gradient 문제 발생

#### **2. Scaled Dot-Product Attention**
- 'Attention Score의 극단화' 해결을 위해 새롭게 제안된 트랜스포머 방식의 표준
- <span style="background-color:pink">Dot-Product Attention 방법대로 내적 진행 후, 값의 크기를 맞추는 **스케일링 과정**이 추가됨</span>
  1. **쿼리(Q) 벡터와 모든 키(K) 벡터를 각각 내적**하여 **유사도** 계산을 진행
        → Attention Score = Q · K    
  2. 계산된 값을 $\sqrt{d_K}$ 로 나누어 **스케일링**을 진행
  3. 스케일링을 진행한 값에 **Softmax** 함수를 적용하여 총합이 1인 **Attention Weight** 산출
- 따라서 스케일링을 통해 계산된 내적값이 과도하게 증가하는 것을 방지 -> 안정적인 분포를 형성 <br>-> Softmax 함수가 안정적으로 학습 진행 가능!
- Scaled Dot-Product Attention 계산의 결과로 얻어지는 벡터
  - **Cross-Attention** : 현재 생성 중인 단어가, 입력 문장에서 어떤 단어들에 집중해야 하는지를 반영한 **Context Vector**
  - **Self-Attention** : 문장 내 다른 단어들과의 관계를 반영한 **문맥적 표현**

## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> 3. Multi-Head Attention </span>
> Transformer 아키텍처의 어느 부분에서 Multi-Head Attention이 사용될까?<br>
- 인코더 : 한 번 사용됨
- 디코더 : 두 번 사용됨
    - **Masked Multi-Head Attention** : 현재까지 생성된 단어들만 참고 (미래 단어를 보지 못하게 가림)
    - **Encoder-Decoder Attention** : 디코더가 출력 생성 시 인코더에 이해한 입력 문장 정보를 참고

<br>=> <span style="background-color:pink">Transformer가 토큰들 사이의 **관계를 파악**하고, **문맥 정보를 반영**해 더 정확한 출력을 만들 수 있게 도와주는 도구!</span>

### <span style="background-color:#e6fae0; color:#42c015; pa?dding:4px; border-radius:5px;"> 3.1 필요성 </span>
#### **1. Single-Head Attention**
1. **개념** : 문장 내 단어들 간의 관계를 파악하기 위해, **하나의 가중치 행렬만을 학습**하고 번역에 사용!
2. **작동방식**
      1. 512차원의 입력 벡터 -> 이 벡터를 위한 어텐션 가중치 분포 계산 -> 가중합 해 새로운 512차원의 벡터 출력
3. **한계**
   1. 하나의 어텐션 분포 안에서 모든 종류의 정보를 한 번에 담아 가중평균을 구함 -> 중요한 관계를 놓칠 수도 있음

#### **2. Multi-Head Attention**
1. **개념** : <span style="background-color:pink">한 단어와 다른 단어간의 관계를 **여러 차원으로 나눠 병렬**로 학습</span>
2. **예시** (Attention is all you need) : 512차원의 입력 벡터를 64차원씩 8개의 벡터로 나눔<br> -> **Attention 스코어를 병렬로 여러 번 계산**

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 3.2 작동방식 </span>
> **예시 상황** : *"**I love you**"* 문장에서, *"**love**"* 라는 단어가 **Multi-Head Attention layer**를 통과하는 과정<br>
> **가정** : <span style="color:green">각 단어의 차원 = 512차원 | 어텐션 세드의 수 = 8개 | 각 헤드의 차원 = 64차원

1. <span style="background-color:yellow">**분할** (projection) : 8명의 헤드에 단어 벡터를 할당</span>
   - 'love'라는 단어를 나타내는 512차원 벡터 하나가 들어옴
   - multi-head attention은 이 벡터를 8개의 작은 벡터 그룹 (Q,K,V)로 투영 -> 서로 다른 8개의 관점으로 분할!
   - 예 : 입력 love_vector(크기 : 1*512)
     - 이 입력 벡터에 8세트의 서로 다른 가중치 행렬을 곱! -> **8개의 독립적인 64차원짜리 작은 벡터들(Q,K,V)로 분할**
       - `Query_1` = `love_vector` × $W_1^Q$ (결과 크기: 1×64)
       - `Key_1` = `love_vector` ×$W_1^K$(결과 크기: 1×64)
       - `Value_1` = `love_vector` × $W_1^V$(결과 크기: 1×64)<br>    …
       - `Query_8` = `love_vector` ×$W_8^Q$ (결과 크기: 1×64)
       - `Key_8` = `love_vector` ×$W_8^K$ (결과 크기: 1×64)
       - `Value_8` = `love_vector` ×$W_8^V$ (결과 크기: 1×64)

2. <span style="background-color:yellow">**병렬 어텐션 계산** : 각 헤드들이 각자 분석 수행</span>
   - 8개의 헤드는 서로 간섭 X -> 병렬로 Scaled Dot-Product Attention 계산
   - `헤드 1`에서 일어나는 일을 살펴보자!
     - **점수 계산** : love의 `Query_1` 벡터를 문장 내 모든 단어 ("I", "love", "you")의 `Key_1`벡터와 내적해 관련성 점수 계산
       - Score_1 = `Query_1` * (`Key_I_1`, `Key_love_1`, `Key_you_1`)

     - **크기 조절** : 계산된 점수들을 각 헤드 차원(64)의 제곱근(8)로 나누어줌

     - **가중치 변환 (Softmax)** : 조절된 점수들에 Softmax 함수를 적용 -> 총합이 1인 `Attention_Weights_1`을 생성
       - 헤드1의 관점에서 love가 다른 단어들에 얼마나 집중해야하는지를 나타냄

     - **가중합** : 가중치를 문장 내 모든 단어의 `Value_1` 벡터에 곱한 뒤 모두 더함
       - `Attention_Output_1` = Attention_Weights_1 * (`Value_I_1`, `Value_love_1`, `Value_you_1`)
       - Attention_Output_1 : 헤드 1의 관점에서 문맥을 이해한 64차원의 결과 벡터
     - 헤드 2, 헤드 3 ... 에서도 각자의 Query, Key, Value를 가지고 동시에 독립적으로 계산 진행

3. <span style="background-color:yellow">**결합 및 최종 투영** : 8개의 헤드가 각자 내놓은 8개의 분석 결과(`Attention_Output_n`)를 하나로 합치기</span>
   - 결합 (Concatenate)
     - 64차원 벡터들을 순서데로 이어붙임 -> 거대한 하나의 512차원 벡터 생성
       - `Concat_Output` = Concat(`Attention_Output_1`, `Attention_Output_2`, ..., `Attention_Output_8`)
   - 최종 투명 (Final Projection)
     - 현재 이 512차원 벡터는 각 헤드의 분석 결과가 나열만 된 상태 <br>-> 정보들을 잘 융합+정리하기 위해 또 다른 가중치 행렬 곱함
       - `Final_Output` = `Concat_Output` × $W^O$
       - `Final_Output` = Multi-Head Attention 레이어의 최종 출력 벡터

## <span style="background-color:#fae0f0; color:#c0156d; padding:4px; border-radius:5px;"> 4. Transformer 아키텍처 </span>

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 4.1 트랜스포머 전처리 단계 </span>

#### **1. 토큰화**
> **입력 텍스트**를 모델이 처리할 수 있는 **단위(tokens)**로 나누는 첫번째 단계<br>

<span style="color:green">**[토큰화 방법]**</span>
- 대부분의 경우 각 토큰은 하나의 단어에 해당 (Write, A, Story)
- 문장 부호 역시 별도의 토큰 (.,!,?)
- 일부 단어는 하나 이상의 토큰으로 나뉠수도 O (doesn't -> does, n't...)

#### **2. 임베딩**
> 토큰화된 각 단위(인간의 언어)는, 임베딩 단계에서 **숫자의 벡터**(컴퓨터의 언어)로 변환됨 <br>
- **임베딩의 구조** : 임베딩은 많은 숫자로 이루어진 긴 벡터로 단어를 나타냄 (벡터의 길이 = 벡터의 차원)
- **임베딩의 원리** : 단어의 의미를 **공간적**으로 표현함
  - <span style="color:blue">유사한 단어 => 유사한 숫자</span>
  - 즉, 의미적으로 유사한 단어들은 임베딩 공간에서 유사한 숫자(좌표)로 변환되어야 함
  - 따라서 어텐션 메커니즘 작동 전에, 단어에 대한 **기본적인 속성을 제공**할 수 있음<br>
- <span style="color:green">**Word2Vec**</span> : 일반적인 임베딩 방법중 하나, 단어를 벡터로 변환한다는 의미
  - **임베딩 생성 원리**
    - **신경망**에 문장을 입력, 다음에 올 단어를 **예측하도록 훈련**
    - 신경망에 있는 **여러 층들**이 점점 더 단어의 **깊은 속성**을 속성 이해하려 함
    - 벡터 추출 : 예측을 수행하는 신경망의 **마지막에서 두번째 레이어에서 나오는 숫자**들 -> **해당 단어의 임베딩**으로 사용됨
      - 마지막에서 두 번째 레이어는 단어의 **매우 심층적인 속성을 포착**해야하므로, 여기서 추출된 벡터는 단어를 잘 설명하는 임베딩이 됨

#### **3. Positional Encoding**
> 단어들의 **순서 정보**를 **임베딩 벡터에 추가**하는 단계 <br>

1. **필요성**
   - 문제 : 트랜스포머는 병렬로 단어를 처리하기 떄문에 단어의 순서를 모름 -> 순서가 달라져도 똑같은 것으로 인식함
   - <span style="background-color:yellow">이 **순서 정보(위치 정보)를 제공**하기 위해 **Positional Encoding** 사용!</span>

2. **동작 방법**
   - 순서 정보 기록 : 각 단어의 임베딩 좌표에 **일관된 순서를 따르는 다른 숫자들을 추가**
     - 일관된 순서? : 문장의 내용이나 단어의 종류와 관계없이, 특정 위치(첫 번째, 두 번째 등)마다 항상 정해진 동일한 위치 벡터를 더해주는 규칙
   - 같은 단어라도 문장에서 차지하는 **위치가 다르면, 다른 좌표**(수정된 임베딩)을 갖게 됨 -> 모델이 순서를 학습할 수 O!

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 4.2 인코더&디코더 </span>

#### **1. Encoder**
1. **역할** : <span style="background-color:pink">입력 문장을 이해하고 요약된 **의미 벡터로 변환**</span>
2. **구성** : **전처리 과정** (<span style="color:blue">토큰화, 임베딩, Positional Encoding</span>) + **인코더 레이어**(<span style="color:blue">Multi-Head Attention, Feed Forward Layer</span>)<br>
3. **다른 모델의 인코더와 비교**

| 구분 | RNN/LSTM | Transformer |
| --- | --- | --- |
| 처리 방식 | 단어를 **순서대로** 입력 | 단어를 **한 번에** 입력 |
| 정보 전달 | 이전 시점의 은닉상태 hₜ₋₁가 다음 시점으로 전달됨 | **Self-Attention**으로 모든 단어가 서로의 관계를 동시에 계산 |
| 순서 인식 | 시점(t) 자체에 **순서가 내재**되어 있음 | 순서가 없기 때문에 **Positional Encoding** 으로 보완 |
| 구조 구성 | RNN / LSTM 셀 반복 | Multi-Head Attention + Feed Forward층 반복 |
| Context Vector | 마지막 단어의 은닉 상태 하나 (hₙ) | 모든 단어의 출력 벡터 전체 |

4. **인코더 레이어의 구성**
   1. **<span style="background-color:yellow">Multi-Head Attention</span>**
   2. **<span style="background-color:yellow">Feed-Forward(FNN)</span>** : 입력 벡터의 차원을 확장, 비선형 변환을 적용해 새로운 표현을 생성하는 신경망 구조
      1. MLP가 2중으로 완전 연결된 구조
      2. **Attention**에서 어떤 단어를 더 집중해서 볼지 정함 -> **Feed-Forward**에서는 그 단어에 대해 **심층 분석** 진행
      3. 아키텍처 속 Add & Norm
         1. **잔차 연결** : 입력 값을 그대로 출력에 더하기
            1. CNN모델 중 ResNet 모델에서 나온 방법으로,<br> 출력에 입력값을 바로 전달함으로써 기울기 소실/폭주 현상을 해결하기 위한 방법
         2. **층 정규화** (layer normalization) : 입력 벡터의 평균, 분산을 사용해 정규화
            1. 데이터의 스케일을 동일하게 만들어, 각 피처들의 중요도를 동증하게 만들어줌<br> -> 학습 안정화 + 모델이 빠르게 수렴하게 됨

#### **2. Decoder**
1. **역할** : <span style="background-color:yellow">인코더가 분석한 입력 문장의 의미 벡터를 받아 **출력 문장을 순차적으로 생성**</span>
2. **구성** : 전처리 과정, 디코더 레이어, 선형 레이어, 소프트맥스 레이어
3. **인코더 vs 디코더 특징 비교**

| **정보 흐름** | **양방향:** 입력 시퀀스 전체를 한 번에 보고 문맥 파악 | **단방향:** 특정 시점의 예측은 과거 토큰에만 의존 |
| --- | --- | --- |
| **병렬 처리** | 셀프 어텐션 후 FFN 단계에서 **병렬 처리**가 가능 → 처리 속도 빠름 | 학습 시에는 병렬 처리이지만, 출력 생성에서 **순차적** → 속도 느림 |
| **입력 및 출력 차원** | 입출력 벡터의 차원은 동일하게 유지됨 | 입출력 벡터의 차원은 동일하게 유지됨 |

4. **디코더 레이어의 구성**
   1. **Masked Multi-Head Attention**
      - 미래 시점의 단어 정보를 참고하지 못하게 마스크를 적용하는 어텐션 메커니즘
        - 트랜스포머 디코더는 시퀀스를 `왼->오 방향`으로 생성
        - 모델이 **과거 정보**(이미 생성된 것)만을 기반으로 다음 단어를 예측하도록 함
        - 따라서 **Look-Ahead Mask**(Causal Mask)를 사용해<br> 현재 단어보다 오른쪽(미래)에 있는 토큰의 어텐션 값을 0으로 만들어 무시

   2. **Encoder-Decoder Attention (Cross-Attention)**
      - 디코더가 인코더의 출력을 참고 + 현재 생성 중인 단어를 입력 문장의 의미와 연결하는 과정
      - **벡터 종류**
        - `Query` : 디코더의 이전 서브레이어에서 출력
        - `Key`, `Value` : 인코더의 최종 출력 (Context Vector)
      - 이를 통해 디코더가 <span style="color:blue">입력 문장 중 어떤 부분에 주목해야하는지</span>를 학습함
      - **예시**
        - "*I go to school*" 에서, **'*school*'**을 생성할 차례일 때,
          - **Query** : "*I go to*" 까지의 문맥을 담은 디코더 상태
          - **Key, Value** : 인코더가 "*나는 학교에 간다*"를 인코딩한 벡터
        - **Cross Attention**은, "*School*"을 만들 때 **학교 부분에 높은 가중치를 부여**<br>+ 한국어 입력의 의미적 정보와 영어 출력의 **대응**을 형성

   3. **Feed=Forward Layer**

### <span style="background-color:#e6fae0; color:#42c015; padding:4px; border-radius:5px;"> 4.3 전체 모델 & 데이터 흐름 </span>
> *I love you*를 트랜스포머로 번역해보자!<br>

#### 1. **입력 준비**
- 문장을 의미+위치 벡터로 변환
  - **임베딩** : 각 단어의 의미를 담은 숫자 벡터로 변환
  - **포지셔널 인코딩** : 각 단어의 문장 속 순서(위치)에 해당하는 벡터를, 의미 벡터에 더함
- 따라서, <span style="background-color:yellow">인코더의 입력 = **의미** + **위치 정보**를 모두 갖춘 벡터!</span>

#### 2. **인코더**
- 입력 문장의 문맥적 의미 깊이 이해하기
- `Self-Attention`을 이용해 입력 문장 내 **모든 단어 간의 관계**를 파악 -> N번 반복 -> 문장 완벽 이해
  1. **셀프 어텐션** : 각 단어의 Q를 만들고, 다른 단어들의 K, V를 참조해 연관성 점수 계산
  2. **관계점수** 계산 : Q-K 간 내적으로 단어 간 관계 점수 산출 -> Softmax로 확률 분포(가중치)로 변환 (각 단어마다 가중치 O)
  3. **가중합** : 계산된 가중치를 V 벡터에 곱 -> 모두 더해 새로운 벡터 생성
  4. **멀티 헤드** : 어떤 헤드는 S-V 관계, 어떤 헤드는 V-O 관계에 집중 -> 다양한 관점에서 문맥 파악
- 따라서, 인코더의 최종 출력은 "I love you"의 Context Vectors(문맥 의미를 모두 반영한 벡터들의 집합)<br> -> 디코더에서  $K_{enc}$, $V_{enc}$로 활용

#### 3. **디코더**
- `<start> `토큰을 시작으로,  인코더의 이해($K_{enc}$, $V_{enc}$)를 참조하며 한 단어씩 출력 생성, `<end>`로 종료
- 각 단계의 출력값이 다음 단계의 입력값으로 투입되며, 다음 단어를 예측